# ⚡ 01. Exploratory Data Analysis & Time-Series Cleaning

**Dataset**: PJM East (PJME) Hourly Electricity Consumption (2002–2018)
**Objective**: Ingest, diagnose missing timestamps, resolve duplicates, and run ADF stationarity tests.

In [ ]:
# Environment Setup
!pip install -q pandas numpy matplotlib seaborn statsmodels

import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

# Direct raw dataset download
DATA_URL = "https://raw.githubusercontent.com/archd3sai/Hourly-Energy-Consumption-Prediction/master/PJME_hourly.csv"
urllib.request.urlretrieve(DATA_URL, "PJME_hourly.csv")
print("✅ PJME_hourly.csv downloaded successfully.")


In [ ]:
# Data Ingestion and Timestamp Normalization
df = pd.read_csv("PJME_hourly.csv")
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').drop_duplicates(subset=['Datetime'])
df = df.set_index('Datetime')

# Check continuous hourly frequency
full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
print(f"Total observed timestamps: {len(df):,}")
print(f"Expected timestamps:        {len(full_range):,}")
print(f"Missing steps:              {len(full_range) - len(df)}")

# Interpolate any missing hours
df = df.reindex(full_range)
df['PJME_MW'] = df['PJME_MW'].interpolate(method='time')
print("✅ Continuous hourly index established.")
df.head()


In [ ]:
# Augmented Dickey-Fuller (ADF) Stationarity Test
adf_result = adfuller(df['PJME_MW'].dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value:       {adf_result[1]:.4e}")
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"   {key}: {value:.4f}")

if adf_result[1] < 0.05:
    print("-> Result: Reject H0; series exhibits mean-reverting stationarity.")
else:
    print("-> Result: Series exhibits unit-root non-stationarity.")


In [ ]:
# Visualize Seasonality (Daily & Annual)
plt.figure(figsize=(14, 5))
plt.plot(df.index[-24*14:], df['PJME_MW'][-24*14:], color='#00d4ff', label='Last 14 Days Demand')
plt.title('PJM Grid Demand (2-Week Sample) — Intraday & Weekly Periodicity')
plt.xlabel('Timestamp')
plt.ylabel('MW Demand')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
